# Train Custom "NEXUS" Wake Word Model with openWakeWord

This notebook trains a custom wake word model for the word **"nexus"** using the openWakeWord framework.

**Runtime**: ~1 hour on Google Colab free T4 GPU  
**Output**: `nexus.onnx` — a small (~800KB) ONNX model file

## How it works
1. Generates synthetic "nexus" audio clips using Piper TTS (multi-speaker, varied accents)
2. Generates adversarial negative examples (similar-sounding words)
3. Downloads background noise data for augmentation
4. Trains a small DNN classifier on top of a frozen embedding model
5. Exports to ONNX format

## Instructions
1. Runtime → Change runtime type → T4 GPU
2. Run all cells in order
3. Download `nexus.onnx` from the final cell

In [ ]:
# === Cell 1: Environment Setup ===
# Install piper-sample-generator for synthetic TTS data
!git clone https://github.com/rhasspy/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!pip install piper-phonemize
!pip install webrtcvad

# Install openwakeword (full installation for training)
!git clone https://github.com/dscripka/openwakeword
!pip install -e ./openwakeword

# Install other training dependencies
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.11.0
!pip install acoustics==0.2.6
!pip install onnx_tf==1.10.0
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19
!pip install scipy

# Download required openWakeWord models
import os
os.makedirs("./openwakeword/openwakeword/resources/models", exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.tflite -O ./openwakeword/openwakeword/resources/models/embedding_model.tflite
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/openwakeword/resources/models/melspectrogram.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.tflite -O ./openwakeword/openwakeword/resources/models/melspectrogram.tflite

print("✅ Environment setup complete!")

In [ ]:
# === Cell 2: Download Training Data ===
import os
import numpy as np
import scipy
from pathlib import Path
from tqdm import tqdm
import datasets

# 1. Download room impulse responses (MIT)
output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)
for row in tqdm(rir_dataset, desc="Downloading RIRs"):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 2. Download noise/background audio (AudioSet - one part)
if not os.path.exists("audioset"):
    os.mkdir("audioset")
fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset, desc="Converting AudioSet"):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

# 3. Download pre-computed openWakeWord features for training and validation
# Training set (~2,000 hours from ACAV100M Dataset)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
# Validation set (~11 hours)
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

print("✅ Training data downloaded!")

In [ ]:
# === Cell 3: Generate Synthetic "nexus" Audio with Piper TTS ===
import os
import sys
sys.path.insert(0, "piper-sample-generator")

# Generate synthetic clips of the word "nexus"
# Using Piper TTS with the LibriTTS multi-speaker model
target_phrase = "nexus"

# Create output directories
os.makedirs("./generated_nexus", exist_ok=True)
os.makedirs("./generated_nexus_augmented", exist_ok=True)

# Generate 5000 positive examples of "nexus"
# Piper generates varied speech with different speakers
print(f"Generating synthetic audio for '{target_phrase}'...")

# Run piper sample generator
!cd piper-sample-generator && python3 generate_samples.py \
    --model models/en_US-libritts_r-medium.pt \
    --text "nexus" \
    --output_dir ../generated_nexus \
    --batch_size 50 \
    --noise_scale 0.8 \
    --noise_scale_w 0.6 \
    --length_scale 0.8 \
    --num_samples 5000

print(f"✅ Generated {len(os.listdir('./generated_nexus'))} synthetic 'nexus' clips")

# Also generate adversarial negatives (similar-sounding words)
adversarial_words = [
    "next", "next us", "nixis", "mexic", "nexus", "necess",
    "lexis", "nixes", "nixus", "noxus", "naxus",
    "text", "taxes", "nexus", "focus", "bonus",
    "census", "versus", "hocus", "locus"
]

os.makedirs("./generated_adversarial", exist_ok=True)
for word in adversarial_words:
    !cd piper-sample-generator && python3 generate_samples.py \
        --model models/en_US-libritts_r-medium.pt \
        --text "{word}" \
        --output_dir ../generated_adversarial \
        --batch_size 50 \
        --noise_scale 0.8 \
        --noise_scale_w 0.6 \
        --length_scale 0.8 \
        --num_samples 250

print(f"✅ Generated {len(os.listdir('./generated_adversarial'))} adversarial negative clips")

In [ ]:
# === Cell 4: Create Training Configuration YAML ===
import yaml

config = {
    "target_phrase": "nexus",
    
    # Audio augmentation
    "augment_data": True,
    "room_impulse_paths": "./mit_rirs",
    "noise_paths": "./audioset_16k",
    
    # Synthetic data paths
    "positive_data_path": "./generated_nexus",
    "adversarial_data_paths": ["./generated_adversarial"],
    
    # Pre-computed features for negative data
    "feature_data_files": {
        "ACAV100M_sample": "./openwakeword_features_ACAV100M_2000_hrs_16bit.npy"
    },
    "false_positive_validation_data_path": "./validation_set_features.npy",
    
    # Training parameters
    "batch_size": 1024,
    "batch_n_per_class": {
        "ACAV100M_sample": 1024,
        "adversarial_negative": 50,
        "positive": 50
    },
    
    # Model architecture
    "model_type": "dnn",
    "layer_size": 32,
    
    # Training schedule
    "max_steps": 50000,
    "learning_rate": 0.001,
    
    # Auto-training controls
    "max_negative_weight": 100,
    "target_false_positives_per_hour": 0.5,
    
    # Output
    "output_dir": "./nexus_model",
    "save_step": 5000,
    
    # TTS batch size for additional generation
    "tts_batch_size": 50,
    "piper_sample_generator_path": "./piper-sample-generator",
}

with open("nexus_config.yml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("✅ Training configuration saved to nexus_config.yml")
print(yaml.dump(config, default_flow_style=False))

In [ ]:
# === Cell 5: Train the Model ===
# This runs the openWakeWord automated training script
# It will take ~30-60 minutes on a T4 GPU

import os
os.chdir("./openwakeword")

# Run the training script with our config
!python train.py --config ../nexus_config.yml

os.chdir("..")
print("✅ Training complete!")

In [ ]:
# === Cell 6: Convert to ONNX and Download ===
import os
import shutil

# Find the best model from training output
model_dir = "./nexus_model"
if os.path.exists(model_dir):
    # Look for the best checkpoint
    models = [f for f in os.listdir(model_dir) if f.endswith(".onnx")]
    if not models:
        # Try looking in openwakeword/models
        alt_dir = "./openwakeword/models"
        if os.path.exists(alt_dir):
            models = [f for f in os.listdir(alt_dir) if f.endswith(".onnx")]
            model_dir = alt_dir
    
    if models:
        print(f"Found models: {models}")
        # Copy the best model
        best_model = sorted(models)[-1]  # usually the last checkpoint is best
        src = os.path.join(model_dir, best_model)
        dst = "./nexus.onnx"
        shutil.copy2(src, dst)
        print(f"✅ Model saved as nexus.onnx ({os.path.getsize(dst)} bytes)")
        print(f"\nDownload this file from the Colab file browser (left sidebar → Files)")
    else:
        print("❌ No .onnx model found. Check training output above for errors.")
        print("Looking in all subdirectories...")
        for root, dirs, files in os.walk("."):
            for f in files:
                if "nexus" in f.lower() and f.endswith(".onnx"):
                    print(f"  Found: {os.path.join(root, f)}")
else:
    print("❌ Model directory not found. Check training output above for errors.")
    # Search for any recently created .onnx files
    for root, dirs, files in os.walk("."):
        for f in files:
            if f.endswith(".onnx") and "nexus" in f.lower():
                print(f"  Found: {os.path.join(root, f)}")

## Next Steps

1. **Download `nexus.onnx`** from the Colab file browser (left sidebar → Files → right-click → Download)
2. **Place it at**: `src-tauri/resources/oww/nexus.onnx`
3. The Rust backend will automatically load it and use it for wake word detection

## What the model does
- Listens to 16kHz mono audio in 1280-sample (80ms) chunks
- Runs a 3-stage pipeline: melspectrogram → embedding → classifier
- Outputs a probability 0.0-1.0 that "nexus" was spoken
- If probability > threshold for multiple consecutive frames → wake triggered
- **No VAD needed** — runs continuously on every audio chunk
- **No ASR needed** — directly detects the acoustic pattern of "nexus"
- Expected: **>95% recall** (vs ~30% with VAD+ASR)